# 💊 Dose-Safe Translator
**Track:** AI + Languages · Veritas AI International High School Hackathon 2026

**Who it's for:** a pharmacist at the counter whose patient reads only Vietnamese, Tagalog,
Haitian Creole, Chinese or Spanish, and needs to understand the directions on an
over-the-counter medicine.

**What goes in → what comes out:** an English medicine instruction such as
*"do not take more than 6 caplets in 24 hours"* → a translation, that translation turned back
into English, and a ✅ / ⚠️ verdict saying whether a person should double-check it.

**Our question:** free AI translators sometimes make dangerous mistakes with medicine
directions, and people often check a translation by translating it back into English.
**Does that back-translation check actually catch the dangerous mistakes, and can targeted
safety checks catch more?**

**This project is a prototype and is not medical advice.**

## How to run this notebook
1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
   The first run downloads the translation model (about 2.5 GB) and takes roughly 10–15 minutes.
2. In section 2, set `TEAM_LANGUAGE` to the language a teammate reads fluently.
3. Section 8 makes a rating sheet. Your teammate rates each translation; upload the sheet and
   re-run section 8 to get the real results (missed dangerous errors and false alarms).

In [ ]:
# 1 · Setup ───────────────────────────────────────────────────────────────
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def ensure_packages(packages):
    """Install any missing packages (Colab already has most of them)."""
    missing = [pip_name for module, pip_name in packages if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


ensure_packages([("transformers", "transformers"), ("sentencepiece", "sentencepiece"),
                 ("sacrebleu", "sacrebleu"), ("gradio", "gradio")])

import random
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
import sacrebleu
import torch
from IPython.display import Markdown, display

QUICK = os.environ.get("DOSESAFE_QUICK") == "1"  # tiny run, only used for testing the notebook
DEVICE = os.environ.get("DOSESAFE_DEVICE") or ("cuda" if torch.cuda.is_available() else "cpu")
random.seed(0)
pd.set_option("display.max_colwidth", 140)
print(f"Running on: {DEVICE.upper()}" + ("  (quick test mode)" if QUICK else ""))
if DEVICE == "cpu" and not QUICK:
    print("Tip: no GPU found. In Colab use Runtime → Change runtime type → T4 GPU (much faster).")

In [ ]:
# 2 · Settings ────────────────────────────────────────────────────────────
MODEL_NAME = "facebook/nllb-200-distilled-600M"  # free translation model from Meta (200 languages)

# For each language: the model's language code, words that mean "not / don't", number words
# (so the checks know "dos" means 2), and words for "under / less" and "over / more" limits.
# To add a language, copy an entry and fill it in (ask a fluent speaker to check the words).
# "flores" is the code used by the FLORES+ test set in section 9 (optional).
LANGUAGES = {
    "Spanish": {
        "code": "spa_Latn", "flores": ["spa_Latn"],
        "negations": ["no", "nunca", "jamás", "ni", "tampoco"],
        "numbers": {"cero": 0, "uno": 1, "una": 1, "un": 1, "dos": 2, "tres": 3, "cuatro": 4,
                    "cinco": 5, "seis": 6, "siete": 7, "ocho": 8, "nueve": 9, "diez": 10,
                    "once": 11, "doce": 12, "quince": 15, "veinte": 20, "veinticuatro": 24,
                    "treinta": 30, "media": 0.5, "medio": 0.5},
        "limits": {"under/less": ["menos", "menor", "menores", "bajo", "inferior", "inferiores"],
                   "over/more": ["más", "mayor", "mayores", "exceder", "exceda", "excede",
                                 "superior", "superiores", "máximo", "máxima"]},
    },
    "Vietnamese": {
        "code": "vie_Latn", "flores": ["vie_Latn"],
        "negations": ["không", "đừng", "chớ", "cấm"],
        "numbers": {"một": 1, "hai": 2, "ba": 3, "bốn": 4, "tư": 4, "năm": 5, "sáu": 6, "bảy": 7,
                    "tám": 8, "chín": 9, "mười": 10, "mười một": 11, "mười hai": 12,
                    "mười lăm": 15, "hai mươi": 20, "hai mươi bốn": 24, "hai mươi tư": 24,
                    "ba mươi": 30, "nửa": 0.5},
        "limits": {"under/less": ["dưới", "ít hơn", "nhỏ hơn", "chưa đủ", "chưa tới"],
                   "over/more": ["trên", "quá", "hơn", "tối đa", "vượt", "trở lên"]},
    },
    "Tagalog": {
        "code": "tgl_Latn", "flores": ["fil_Latn", "tgl_Latn"],
        "negations": ["huwag", "hindi", "wag", "di"],
        "numbers": {"isa": 1, "isang": 1, "dalawa": 2, "dalawang": 2, "tatlo": 3, "tatlong": 3,
                    "apat": 4, "lima": 5, "limang": 5, "anim": 6, "pito": 7, "pitong": 7,
                    "walo": 8, "walong": 8, "siyam": 9, "sampu": 10, "sampung": 10,
                    "labing-isa": 11, "labindalawa": 12, "labindalawang": 12, "labinlima": 15,
                    "dalawampu": 20, "dalawampu't apat": 24, "kalahati": 0.5},
        "limits": {"under/less": ["ilalim", "ibaba", "wala pang", "pababa", "mas bata", "kulang",
                                  "mababa", "hindi pa"],
                   "over/more": ["higit", "mahigit", "pataas", "lampas", "lumampas", "sobra",
                                 "mas matanda", "pinakamarami", "hanggang"]},
    },
    "Haitian Creole": {
        "code": "hat_Latn", "flores": ["hat_Latn"],
        "negations": ["pa", "pinga", "jamè", "okenn"],
        "numbers": {"youn": 1, "en": 1, "de": 2, "twa": 3, "kat": 4, "senk": 5, "sis": 6,
                    "sèt": 7, "uit": 8, "nèf": 9, "dis": 10, "onz": 11, "douz": 12, "kenz": 15,
                    "ven": 20, "vennkat": 24, "trant": 30, "mwatye": 0.5, "demi": 0.5},
        "limits": {"under/less": ["anba", "mwens", "pi piti", "pi jèn", "poko"],
                   "over/more": ["plis", "depase", "pi gran", "anwo", "maksimòm", "pase"]},
    },
    "Chinese (Simplified)": {
        "code": "zho_Hans", "flores": ["cmn_Hans", "zho_Hans"], "cjk": True,
        "negations": ["不", "勿", "别", "没", "禁"],
        "numbers": {"一": 1, "二": 2, "两": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8,
                    "九": 9, "十": 10, "十一": 11, "十二": 12, "十五": 15, "二十": 20,
                    "二十四": 24, "三十": 30, "半": 0.5},
        "limits": {"under/less": ["以下", "不满", "未满", "少于", "低于", "小于", "不到"],
                   "over/more": ["以上", "超过", "多于", "高于", "大于", "最多", "超出"]},
    },
}
ENGLISH = {
    "code": "eng_Latn",
    "negations": ["not", "never", "no", "nor", "cannot", "don't", "dont", "doesn't"],
    "numbers": {"zero": 0, "one": 1, "once": 1, "two": 2, "twice": 2, "three": 3, "four": 4,
                "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11,
                "twelve": 12, "fifteen": 15, "twenty": 20, "twenty-four": 24, "twenty four": 24,
                "thirty": 30, "half": 0.5},
    "limits": {"under/less": ["under", "below", "younger", "less", "fewer", "lower"],
               "over/more": ["over", "above", "older", "more", "exceed", "exceeds", "exceeding",
                             "maximum", "most", "longer"]},
}

WIDELY_SERVED = "Spanish"      # a language free translators handle well, for comparison
TEAM_LANGUAGE = "Vietnamese"   # ← change this to the language your teammate reads fluently
EVAL_LANGUAGES = list(dict.fromkeys([WIDELY_SERVED, TEAM_LANGUAGE]))

N_INSTRUCTIONS = 12 if QUICK else 150  # label instructions to test in each language
N_CANDIDATES = 3 if QUICK else 5       # translations to try in "safer mode"
BACK_SIMILARITY_MIN = 40               # back-translation similarity (0–100) below this is flagged
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 3 · Real medicine directions from US drug labels
We use **openFDA** to find over-the-counter labels for common medicines, then download each
label's original file from **DailyMed** (US National Library of Medicine). In those files every
instruction is its own bullet point, so each one becomes one test sentence. We keep the
**Directions** section and the **Do not use** warnings. The result is saved to
`data/otc_directions.csv`, so later runs don't need the internet for this step.

In [ ]:
# 3 · Download and clean the label instructions ───────────────────────────
INGREDIENTS = ["acetaminophen", "ibuprofen", "naproxen sodium", "aspirin", "diphenhydramine",
               "loratadine", "cetirizine", "fexofenadine", "dextromethorphan", "guaifenesin",
               "phenylephrine", "famotidine", "omeprazole", "loperamide", "bismuth subsalicylate",
               "calcium carbonate", "docusate", "sennosides", "polyethylene glycol 3350",
               "doxylamine", "meclizine", "hydrocortisone", "miconazole", "nicotine"]
LABELS_PER_INGREDIENT = 3
DATA_FILE = DATA_DIR / "otc_directions.csv"
HEADERS = {"User-Agent": "dose-safe-hackathon-prototype"}
HL7 = "{urn:hl7-org:v3}"
SECTIONS = {"34068-7": "Directions", "50570-1": "Do not use"}  # label section codes


def find_otc_labels(ingredient, limit):
    """Ask openFDA for over-the-counter labels that contain this ingredient."""
    query = f'openfda.product_type:"HUMAN OTC DRUG" AND openfda.generic_name:"{ingredient}"'
    response = requests.get("https://api.fda.gov/drug/label.json",
                            params={"search": query, "limit": limit}, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        return []
    return [(r["set_id"], (r.get("openfda", {}).get("brand_name") or [""])[0])
            for r in response.json().get("results", []) if r.get("set_id")]


def fetch_label(set_id):
    """Download one label's original XML file from DailyMed."""
    url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code == 200:
                return ET.fromstring(response.content)
        except (requests.RequestException, ET.ParseError):
            pass
        time.sleep(1 + attempt)
    return None


def clean(text):
    text = unicodedata.normalize("NFKC", text).replace("⁄", "/")
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[•●▪■◦]", " ", text)
    return " ".join(text.split()).strip(" ;,-–—:")


def element_text(element, skip_lists=False):
    """All the text inside an XML element (optionally leaving out nested lists and tables)."""
    parts = [element.text or ""]
    for child in element:
        if not (skip_lists and child.tag in (f"{HL7}list", f"{HL7}table")):
            parts.append(element_text(child, skip_lists))
        parts.append(child.tail or "")
    return " ".join(" ".join(parts).split())


def child_lists(element):
    """Lists directly inside this element (not lists nested deeper inside those lists)."""
    found = []
    for child in element:
        if child.tag == f"{HL7}list":
            found.append(child)
        elif child.tag != f"{HL7}table":
            found.extend(child_lists(child))
    return found


def list_units(xml_list, prefix=""):
    """Turn a bulleted list into sentences, e.g. 'adults: take 2 caplets every 6 hours'."""
    units = []
    for item in xml_list.findall(f"{HL7}item"):
        own_text = clean(element_text(item, skip_lists=True))
        label = f"{prefix}: {own_text}" if prefix and own_text else (prefix or own_text)
        nested = child_lists(item)
        if nested:
            for sub_list in nested:
                units += list_units(sub_list, label)
        elif label:
            units.append(label)
    return units


def table_units(table):
    """Dosing tables: 'children 6-11 years: take 1 tablet every 4 hours'."""
    units = []
    for row in table.iter(f"{HL7}tr"):
        if any(cell.tag == f"{HL7}th" for cell in row):
            continue  # header row
        cells = [clean(element_text(cell)) for cell in row if cell.tag == f"{HL7}td"]
        cells = [c for c in cells if c]
        if len(cells) >= 2:
            units.append(f"{cells[0]}: {'; '.join(cells[1:])}")
        elif cells:
            units.append(cells[0])
    return units


def section_units(section, heading):
    units = []
    # A line like "Do not use if you have:" introduces the bullets after it, so we keep it as a lead-in.
    title = section.find(f"{HL7}title")
    title_text = clean(element_text(title)) if title is not None else ""
    lead = title_text if heading == "Do not use" and len(title_text.split()) > 3 else ""
    text_element = section.find(f"{HL7}text")
    if text_element is not None:
        for child in text_element:
            if child.tag == f"{HL7}list":
                units += list_units(child, lead)
            elif child.tag == f"{HL7}table":
                units += table_units(child)
            elif child.tag == f"{HL7}paragraph":
                own_text = element_text(child, skip_lists=True)
                inner_lists = child_lists(child)
                if inner_lists:  # "Do not use if you have <bullets>" in one paragraph
                    paragraph_lead = clean(own_text)
                    if paragraph_lead.lower() in ("", "directions", "do not use"):
                        paragraph_lead = lead
                    for xml_list in inner_lists:
                        units += list_units(xml_list, paragraph_lead)
                elif own_text.rstrip().endswith(":"):
                    lead = clean(own_text)
                else:
                    units += [clean(piece) for piece in re.split(r"[•●▪■]", own_text)]
                for table in child.iter(f"{HL7}table"):
                    units += table_units(table)
    for sub_section in section.findall(f"{HL7}component/{HL7}section"):
        code = sub_section.find(f"{HL7}code")
        if code is None or code.get("code") in (None, "42229-5"):  # only untitled sub-parts
            units += section_units(sub_section, heading)
    if heading == "Do not use":  # turn bullets like "if you have an ulcer" into full warnings
        continuations = ("if ", "with ", "for ", "in ", "on ", "unless ", "while ", "to ")

        def full_warning(u):
            has_own_not = re.search(r"\b(not|never)\b", u.lower())
            if not has_own_not and (u[:1].islower() or u.lower().startswith(continuations)):
                return f"Do not use {u}"
            return u

        units = [full_warning(u) for u in units if u]
    # drop lead-ins that lost their bullets, like "Do not use if you have"
    return [u for u in units if not re.search(r"(if you (have|are)|the following)$", u.lower())]


def build_dataset():
    rows, seen = [], set()
    for ingredient in INGREDIENTS:
        for set_id, brand in find_otc_labels(ingredient, LABELS_PER_INGREDIENT):
            root = fetch_label(set_id)
            time.sleep(0.2)  # be polite to the free government servers
            if root is None:
                continue
            for section in root.iter(f"{HL7}section"):
                code = section.find(f"{HL7}code")
                heading = SECTIONS.get(code.get("code") if code is not None else None)
                if not heading:
                    continue
                for unit in section_units(section, heading):
                    unit = clean(unit)
                    if 4 <= len(unit.split()) <= 60 and unit.lower() not in seen:
                        seen.add(unit.lower())
                        rows.append({"instruction": unit, "section": heading,
                                     "ingredient": ingredient, "brand": brand, "set_id": set_id})
    return pd.DataFrame(rows)


if DATA_FILE.exists():
    data = pd.read_csv(DATA_FILE)
    print(f"Loaded {len(data)} saved instructions from {DATA_FILE}")
else:
    print("Downloading labels from openFDA and DailyMed (about 1–2 minutes)...")
    data = build_dataset()
    data.insert(0, "instruction_id", range(1, len(data) + 1))
    data.to_csv(DATA_FILE, index=False)
    print(f"Saved {len(data)} unique instructions from {data.set_id.nunique()} labels to {DATA_FILE}")
display(data.sample(min(8, len(data)), random_state=1)[["instruction", "section", "ingredient"]])

## 4 · Safety checks: how we spot a dangerous translation
A translation is **flagged** (⚠️ "ask a pharmacist to check") if any of these fail.
The first four look at the **translation itself**; the rest look at the **back-translation**
(the translation turned back into English).

| Check | What it looks for |
|---|---|
| **Numbers kept** | Every number in the English (2 caplets, 6 hours, 12 years) appears in the translation, as digits or words, and no new number appears. |
| **"Not" kept** | The translation has at least as many "not / don't" words as the English. |
| **Under / over kept** | If the English says *under* 12 or *no more than* 6, the translation still has a word for *under* or *more* (Spanish *menores de*, Vietnamese *dưới*, Chinese *以下*, …). |
| **Actually translated** | The model didn't just copy the English. |
| **Numbers survive the round trip** | Every number is still there after translating back into English. |
| **"Not" survives the round trip** | The back-translation still says "not" (or "without"). |
| **Units survive the round trip** | Hours stay hours, days stay days, mg stays mg. |
| **Under / over survives the round trip** | The back-translation still says *under*, *over*, *more*, *less*… |
| **Back-translation matches** | The back-translation is similar to the original (chrF score). This is the check people usually do by eye. |

These are simple word rules, so they can't understand meaning. That is exactly why section 8
tests them against a real speaker's judgement.

In [ ]:
# 4 · Safety checks ─────────────────────────────────────────────────────────
DIGITS = re.compile(r"\d+(?:[.,]\d+)?(?:\s*/\s*\d+)?")
WORDS = re.compile(r"[^\W\d_]+(?:['’-][^\W\d_]+)*")
UNITS = {  # ages ("12 years") are already protected by the numbers check
    "hours": ["hour", "hours", "hr", "hrs", "hourly"],
    "days": ["day", "days", "daily"],
    "weeks": ["week", "weeks", "weekly"],
    "months": ["month", "months", "monthly"],
    "minutes": ["minute", "minutes", "min", "mins"],
    "mg": ["mg", "milligram", "milligrams"],
    "mL": ["ml", "milliliter", "milliliters", "millilitre", "millilitres"],
    "teaspoons": ["teaspoon", "teaspoons", "tsp"],
    "tablespoons": ["tablespoon", "tablespoons", "tbsp"],
}
FORWARD_CHECKS = ["Numbers kept", "“Not” kept", "Under / over kept", "Actually translated"]
ROUND_TRIP_CHECKS = ["Numbers survive the round trip", "“Not” survives the round trip",
                     "Units survive the round trip", "Under / over survives the round trip",
                     "Back-translation matches"]
CHECK_NAMES = FORWARD_CHECKS + ROUND_TRIP_CHECKS


def normalize(text):
    return unicodedata.normalize("NFKC", text).replace("⁄", "/").lower()


def _number_table(spec):
    if "_table" not in spec:
        spec["_table"] = {tuple(WORDS.findall(normalize(w))): v for w, v in spec["numbers"].items()}
    return spec["_table"]


def number_parts(text, spec):
    """Numbers written as digits, and numbers written as words, in a text."""
    text = normalize(text)
    digits = []
    for token in DIGITS.findall(text):
        token = token.replace(" ", "")
        try:
            if "/" in token:
                top, bottom = token.split("/")
                digits.append(round(float(top.replace(",", ".")) / float(bottom), 3))
            else:
                digits.append(round(float(token.replace(",", ".")), 3))
        except (ValueError, ZeroDivisionError):
            pass
    words = []
    if spec.get("cjk"):  # Chinese has no spaces: match the longest number word at each position
        keys = sorted(spec["numbers"], key=len, reverse=True)
        i = 0
        while i < len(text):
            match = next((k for k in keys if text.startswith(k, i)), None)
            if match:
                words.append(float(spec["numbers"][match]))
                i += len(match)
            else:
                i += 1
    else:
        tokens, table = WORDS.findall(text), _number_table(spec)
        longest = max(len(k) for k in table)
        i = 0
        while i < len(tokens):
            for size in range(min(longest, len(tokens) - i), 0, -1):
                key = tuple(tokens[i:i + size])
                if key in table:
                    words.append(float(table[key]))
                    i += size
                    break
            else:
                i += 1
    return digits, words


def count_negations(text, spec):
    text = normalize(text)
    if spec.get("cjk"):
        return sum(text.count(word) for word in spec["negations"])
    negations = set(spec["negations"])
    return sum(1 for t in WORDS.findall(text) if t in negations or t.endswith(("n't", "n’t")))


def word_groups(text, groups):
    tokens = set(WORDS.findall(normalize(text)))
    return {name for name, words in groups.items() if tokens & set(words)}


def limit_groups(text, spec):
    """Which kinds of limit ("under/less", "over/more") a text mentions, in its own language."""
    text = normalize(text)
    if spec.get("cjk"):
        return {name for name, words in spec["limits"].items() if any(w in text for w in words)}
    padded = " " + " ".join(WORDS.findall(text)) + " "
    return {name for name, phrases in spec["limits"].items()
            if any(" " + " ".join(WORDS.findall(normalize(p))) + " " in padded for p in phrases)}


def _show(values):
    return ", ".join(str(int(v)) if float(v).is_integer() else str(v) for v in sorted(values))


def compare_numbers(source, text, spec):
    src_digits, src_words = number_parts(source, ENGLISH)
    needed = Counter(src_digits + src_words)
    digits, words = number_parts(text, spec)
    missing = needed - Counter(digits + words)
    new = Counter(digits) - needed
    problems = []
    if missing:
        problems.append("missing " + _show(missing.elements()))
    if new:
        problems.append("new " + _show(new.elements()))
    return not problems, "; ".join(problems)


def safety_checks(source, translation, back, spec):
    """Run every check. Returns {check name: {"ok": True/False, "detail": "..."}}."""
    results = {}
    n_not = count_negations(source, ENGLISH)
    source_limits = limit_groups(source, ENGLISH)

    # Checks on the translation itself
    ok, detail = compare_numbers(source, translation, spec)
    results["Numbers kept"] = {"ok": ok, "detail": detail}
    n = count_negations(translation, spec)
    results["“Not” kept"] = {"ok": n >= n_not, "detail": f"English has {n_not}, translation has {n}"}
    lost = source_limits - limit_groups(translation, spec)
    results["Under / over kept"] = {"ok": not lost, "detail": "no word for " + ", ".join(sorted(lost))}
    copied = sacrebleu.sentence_chrf(translation, [source]).score
    results["Actually translated"] = {"ok": copied < 75, "detail": f"{copied:.0f}/100 identical to English"}

    # Checks on the back-translation (the translation turned back into English)
    ok, detail = compare_numbers(source, back, ENGLISH)
    results["Numbers survive the round trip"] = {"ok": ok, "detail": detail}
    n = count_negations(back, ENGLISH) + len(re.findall(r"\bwithout\b", normalize(back)))
    results["“Not” survives the round trip"] = {"ok": n >= n_not,
                                                "detail": f"English has {n_not}, back-translation has {n}"}
    lost = word_groups(source, UNITS) - word_groups(back, UNITS)
    results["Units survive the round trip"] = {"ok": not lost, "detail": "lost " + ", ".join(sorted(lost))}
    lost = source_limits - limit_groups(back, ENGLISH)
    results["Under / over survives the round trip"] = {"ok": not lost,
                                                       "detail": "lost " + ", ".join(sorted(lost))}
    similarity = sacrebleu.sentence_chrf(back, [source]).score
    results["Back-translation matches"] = {"ok": similarity >= BACK_SIMILARITY_MIN,
                                           "detail": f"similarity {similarity:.0f}/100"}
    return results


def failed_checks(source, translation, back, spec):
    return [name for name, c in safety_checks(source, translation, back, spec).items() if not c["ok"]]


# Quick self-test with hand-made Spanish examples (translation, back-translation)
_src = "do not take more than 6 caplets in 24 hours"
_tests = {
    "correct": ("no tome más de 6 cápsulas en 24 horas", "do not take more than 6 capsules in 24 hours"),
    "numbers as words": ("no tome más de seis cápsulas en veinticuatro horas",
                         "do not take more than six capsules in twenty-four hours"),
    "lost the 'not'": ("tome más de 6 cápsulas en 24 horas", "take more than 6 capsules in 24 hours"),
    "wrong number": ("no tome más de 8 cápsulas en 24 horas", "do not take more than 8 capsules in 24 hours"),
    "hours → days": ("no tome más de 6 cápsulas en 24 días", "do not take more than 6 capsules in 24 days"),
}
for _name, (_translation, _back) in _tests.items():
    _failed = failed_checks(_src, _translation, _back, LANGUAGES["Spanish"])
    print(f"{_name:17} → " + ("passes" if not _failed else "FLAGGED: " + ", ".join(_failed)))
assert not failed_checks(_src, *_tests["correct"], LANGUAGES["Spanish"])
assert not failed_checks(_src, *_tests["numbers as words"], LANGUAGES["Spanish"])
assert all(failed_checks(_src, *_tests[k], LANGUAGES["Spanish"]) for k in ["lost the 'not'", "wrong number", "hours → days"])

_src = "children under 12 years: ask a doctor"
_correct = ("niños menores de 12 años: consulte a un médico", "children under 12 years: see a doctor")
_dropped = ("niños de 12 años: consulte a un médico", "children 12 years old: see a doctor")
print(f"{'under kept':17} → " + (", ".join(failed_checks(_src, *_correct, LANGUAGES["Spanish"])) or "passes"))
print(f"{'under dropped':17} → FLAGGED: " + ", ".join(failed_checks(_src, *_dropped, LANGUAGES["Spanish"])))
assert not failed_checks(_src, *_correct, LANGUAGES["Spanish"])
assert "Under / over kept" in failed_checks(_src, *_dropped, LANGUAGES["Spanish"])

## 5 · The translation model
**NLLB-200** ("No Language Left Behind") is a free model from Meta that translates between 200
languages, including many that big apps serve poorly. For each instruction we ask it for its
top few translations (beam search), and we translate each one back into English for the checks.

In [ ]:
# 5 · Load the translation model ──────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if DEVICE == "cuda":
    model = model.half()  # half precision: faster on GPU, same translations in practice
model = model.to(DEVICE).eval()


@torch.inference_mode()
def translate(texts, src_code, tgt_code, n_candidates=1, num_beams=4, batch_size=16):
    """Translate a list of texts. Returns, for each text, its top `n_candidates` translations,
    best first (the model's own ranking from beam search)."""
    tokenizer.src_lang = src_code
    num_beams = max(num_beams, n_candidates)
    results = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=256).to(DEVICE)
        output = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
                                num_beams=num_beams, num_return_sequences=n_candidates, max_length=256)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        for i in range(len(batch)):
            results.append(decoded[i * n_candidates:(i + 1) * n_candidates])
    return results


example = "do not take more than 6 caplets in 24 hours"
for language in EVAL_LANGUAGES:
    print(f"{language}: {translate([example], 'eng_Latn', LANGUAGES[language]['code'])[0][0]}")

## 6 · Translate and check every instruction
For each instruction and language we keep two versions:
- **NLLB as-is** – the model's first choice (the baseline anyone would get).
- **Safer mode** – out of the model's top `N_CANDIDATES` translations, the one that fails the
  fewest checks on the **translation itself**, then the fewest checks overall (ties go to the
  model's own ranking). This is our improvement. We rank the translation-side checks first
  because a back-translation can lose a word like "under" even when the translation was right.

In [ ]:
# 6 · Run every instruction through both versions ─────────────────────────
def translate_and_check(texts, language, n_candidates):
    """Translate each English text, translate every candidate back, run the checks, and keep
    the model's first choice ("baseline") and the candidate that passes the most checks ("safer")."""
    spec = LANGUAGES[language]
    candidates = translate(texts, "eng_Latn", spec["code"], n_candidates=n_candidates)
    flat = [translation for options in candidates for translation in options]
    backs = [options[0] for options in translate(flat, spec["code"], "eng_Latn")]
    results, k = [], 0
    for source, options in zip(texts, candidates):
        scored = []
        for rank, translation in enumerate(options):
            checks = safety_checks(source, translation, backs[k], spec)
            scored.append({"rank": rank, "translation": translation, "back": backs[k], "checks": checks,
                           "failed": [name for name, c in checks.items() if not c["ok"]]})
            k += 1
        safer = min(scored, key=lambda option: (
            sum(name in FORWARD_CHECKS for name in option["failed"]),  # problems in the translation itself
            len(option["failed"]),                                     # then problems overall
            option["rank"]))                                           # then the model's own ranking
        results.append({"source": source, "baseline": scored[0], "safer": safer,
                        "changed": safer["rank"] != 0})
    return results


data["has_number"] = data.instruction.map(lambda t: any(number_parts(t, ENGLISH)))
data["has_not"] = data.instruction.map(lambda t: count_negations(t, ENGLISH) > 0)
data["has_limit"] = data.instruction.map(lambda t: bool(limit_groups(t, ENGLISH)))
data["risky"] = data.has_number | data.has_not | data.has_limit
risky, plain = data[data.risky], data[~data.risky]
n_risky = min(N_INSTRUCTIONS, len(risky))
sample_df = pd.concat([risky.sample(n_risky, random_state=0),
                       plain.sample(min(N_INSTRUCTIONS - n_risky, len(plain)), random_state=0)])
print(f"{data.risky.mean():.0%} of the {len(data)} instructions contain a number, a 'not' or an "
      f"age/amount limit. Testing {len(sample_df)} of them in {', '.join(EVAL_LANGUAGES)}.")

rows = []
for language in EVAL_LANGUAGES:
    started = time.time()
    results = translate_and_check(sample_df.instruction.tolist(), language, N_CANDIDATES)
    for instruction_id, result in zip(sample_df.instruction_id, results):
        rows.append({"language": language, "instruction_id": instruction_id, "source": result["source"],
                     "baseline": result["baseline"]["translation"], "baseline_back": result["baseline"]["back"],
                     "baseline_failed": result["baseline"]["failed"],
                     "safer": result["safer"]["translation"], "safer_back": result["safer"]["back"],
                     "safer_failed": result["safer"]["failed"], "changed": result["changed"]})
    print(f"{language}: {len(results)} instructions done in {time.time() - started:.0f} s")
results_df = pd.DataFrame(rows)
results_df.to_csv(DATA_DIR / "results.csv", index=False)

## 7 · Results from the automatic checks
⚠️ Safer mode picks translations *using* these checks, so it will always look better here.
These numbers show where the checks see problems; the real test is section 8.

In [ ]:
# 7 · Summary tables and examples ──────────────────────────────────────────
SYSTEMS = {"baseline": "NLLB as-is", "safer": f"Safer mode (best of {N_CANDIDATES})"}
summary = []
for language, group in results_df.groupby("language", sort=False):
    for system, label in SYSTEMS.items():
        failed = group[f"{system}_failed"]
        row = {"language": language, "version": label, "flagged %": 100 * failed.map(bool).mean()}
        for check in CHECK_NAMES:
            row[check] = 100 * failed.map(lambda names: check in names).mean()
        summary.append(row)
summary = pd.DataFrame(summary).round(1)
display(Markdown("**Share of translations failing each check (%)**"))
display(summary)

for language in EVAL_LANGUAGES:
    group = results_df[results_df.language == language]
    flagged = group[group.baseline_failed.map(bool)].head(5)
    display(Markdown(f"**{language}: translations the checks flagged** (NLLB as-is)"))
    display(pd.DataFrame({"English (label)": flagged.source, language: flagged.baseline,
                          "Back to English": flagged.baseline_back,
                          "Problems found": flagged.baseline_failed.map(", ".join)}).reset_index(drop=True))
    changed = group[group.changed].head(3)
    if len(changed):
        display(Markdown(f"**{language}: where safer mode picked a different translation**"))
        display(pd.DataFrame({"English (label)": changed.source, "NLLB as-is": changed.baseline,
                              "Safer mode": changed.safer,
                              "Safer mode, back to English": changed.safer_back}).reset_index(drop=True))

print("Numbers for the write-up:")
for language in EVAL_LANGUAGES:
    group = results_df[results_df.language == language]
    print(f"- {language}: checks flagged {group.baseline_failed.map(bool).mean():.0%} of NLLB's "
          f"translations; safer mode changed {group.changed.mean():.0%} of them and "
          f"{group.safer_failed.map(bool).mean():.0%} stayed flagged.")

## 8 · The real test: a fluent speaker rates the translations
The automatic checks can't understand meaning, so a teammate who reads `TEAM_LANGUAGE`
fluently rates translations **without knowing which version made them or whether they were
flagged**. Then we can measure honestly:
- how often each version is **dangerous**, and
- how many dangerous translations each checker **misses** (false negatives) and how many
  **false alarms** it raises (false positives): our full checks vs. a back-translation-only check.

**Steps:** run this cell → download `rate_me_<language>.csv` **and** `answer_key_<language>.csv`
from the Files panel (📁 on the left) → your teammate fills in the `rating` column (and
`correct_translation` for any dangerous one) → upload the filled sheet into `data/` named
`rated_<language>.csv` → run this cell again.

In [ ]:
# 8 · Rating sheet and evaluation ──────────────────────────────────────────
RATING_GUIDE = """Rating guide (column `rating`):
  2 = correct: a patient would follow it exactly right
  1 = small mistake, but the meaning and safety are the same
  0 = DANGEROUS: wrong number, dose, time or age limit; the "not" is lost; or confusing enough to be misused"""
slug = TEAM_LANGUAGE.split()[0].lower()
SHEET_FILE, KEY_FILE = DATA_DIR / f"rate_me_{slug}.csv", DATA_DIR / f"answer_key_{slug}.csv"
RATED_FILE = DATA_DIR / f"rated_{slug}.csv"


def make_rating_sheet(language, max_instructions=60):
    group = results_df[results_df.language == language].head(max_instructions)
    items = []
    for _, r in group.iterrows():
        versions = [("baseline", r.baseline, r.baseline_failed)]
        if r.changed:
            versions.append(("safer", r.safer, r.safer_failed))
        for system, translation, failed in versions:
            items.append({"instruction_id": r.instruction_id, "system": system, "english": r.source,
                          "translation": translation, "flag_full": bool(failed),
                          "flag_backonly": "Back-translation matches" in failed})
    items = pd.DataFrame(items).sample(frac=1, random_state=1).reset_index(drop=True)  # shuffle = blind
    items.insert(0, "item_id", range(1, len(items) + 1))
    sheet = items[["item_id", "english", "translation"]].assign(rating="", correct_translation="", notes="")
    return sheet, items[["item_id", "instruction_id", "system", "flag_full", "flag_backonly"]]


def checker_scores(df, column):
    flagged, dangerous = df[column].astype(bool), df.dangerous
    tp, fn = int((flagged & dangerous).sum()), int((~flagged & dangerous).sum())
    fp, tn = int((flagged & ~dangerous).sum()), int((~flagged & ~dangerous).sum())
    return {"dangerous caught": tp, "dangerous MISSED (false negatives)": fn,
            "false alarms (false positives)": fp, "correctly passed": tn,
            "catch rate %": round(100 * tp / (tp + fn), 1) if tp + fn else None,
            "false alarm rate %": round(100 * fp / (fp + tn), 1) if fp + tn else None}


def evaluate_ratings(rated_file, key_file):
    rated, key = pd.read_csv(rated_file), pd.read_csv(key_file)
    df = key.merge(rated[["item_id", "rating"]], on="item_id")
    df["rating"] = pd.to_numeric(df.rating, errors="coerce")
    df = df.dropna(subset=["rating"])
    df["dangerous"] = df.rating == 0
    print(f"{len(df)} rated translations.\n")

    per_version = {"baseline": [], "safer": []}
    for _, group in df.groupby("instruction_id"):
        base = group[group.system == "baseline"]
        if base.empty:
            continue
        safer = group[group.system == "safer"]
        per_version["baseline"].append(bool(base.dangerous.iloc[0]))
        per_version["safer"].append(bool((safer if len(safer) else base).dangerous.iloc[0]))
    display(Markdown("**How often is each version dangerous?**"))
    display(pd.DataFrame([{"version": SYSTEMS[s], "instructions rated": len(v),
                           "dangerous %": round(100 * sum(v) / len(v), 1) if v else None}
                          for s, v in per_version.items()]))

    display(Markdown("**Does each checker catch the dangerous translations?**"))
    display(pd.DataFrame([{"checker": "Back-translation only (what people usually do)",
                           **checker_scores(df, "flag_backonly")},
                          {"checker": "Our full safety checks", **checker_scores(df, "flag_full")}]))
    return df


if RATED_FILE.exists() and KEY_FILE.exists():
    rated_df = evaluate_ratings(RATED_FILE, KEY_FILE)
else:
    sheet, key = make_rating_sheet(TEAM_LANGUAGE)
    sheet.to_csv(SHEET_FILE, index=False)
    key.to_csv(KEY_FILE, index=False)
    print(f"Made a rating sheet with {len(sheet)} translations: {SHEET_FILE}")
    print(f"Answer key (don't show the rater): {KEY_FILE}\n")
    print(RATING_GUIDE)
    print(f"\nWhen it's filled in, upload it as {RATED_FILE} and run this cell again.")

## 9 · (Optional) General translation quality on FLORES+
FLORES+ is a standard test set: the same 1,000 sentences professionally translated into 200+
languages. Scoring NLLB on it shows how much worse it is in general for less widely served
languages. It needs a free Hugging Face account: accept the terms at
https://huggingface.co/datasets/openlanguagedata/flores_plus, create an access token, and add
it in Colab under 🔑 **Secrets** with the name `HF_TOKEN`. Skipped automatically otherwise.

In [ ]:
# 9 · FLORES+ chrF score (optional) ────────────────────────────────────────
def hugging_face_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


def flores_sentences(codes, n, token):
    from datasets import load_dataset
    last_error = None
    for code in codes:
        try:
            return load_dataset("openlanguagedata/flores_plus", code, split="devtest", token=token)["text"][:n]
        except Exception as error:
            last_error = error
    raise last_error


token = hugging_face_token()
if QUICK or not token:
    print("Skipped: " + ("quick test mode." if QUICK else "no Hugging Face token (see the note above)."))
else:
    ensure_packages([("datasets", "datasets")])
    english = flores_sentences(["eng_Latn"], 100, token)
    for language in EVAL_LANGUAGES:
        try:
            references = flores_sentences(LANGUAGES[language]["flores"], 100, token)
        except Exception as error:
            print(f"{language}: couldn't load FLORES+ ({error})")
            continue
        outputs = [options[0] for options in translate(english, "eng_Latn", LANGUAGES[language]["code"])]
        print(f"{language}: chrF {sacrebleu.corpus_chrf(outputs, [references]).score:.1f} / 100")

## 10 · Try it: the Dose-Safe app
Type or pick an instruction, choose a language, and see the translation, the back-translation
and every check. **Prototype – not medical advice.**

In [ ]:
# 10 · Gradio demo app ─────────────────────────────────────────────────────
import gradio as gr


def check_one(english, language, safer_mode):
    english = (english or "").strip()
    if not english:
        return "", "", "Type a medicine instruction in English first."
    result = translate_and_check([english], language, N_CANDIDATES if safer_mode else 1)[0]
    chosen = result["safer"] if safer_mode else result["baseline"]
    lines = []
    for name, check in chosen["checks"].items():
        lines.append(f"{'✅' if check['ok'] else '❌'} **{name}**" + ("" if check["ok"] else f" ({check['detail']})"))
    if chosen["failed"]:
        verdict = "### ⚠️ Ask a pharmacist or interpreter to check this translation"
    else:
        verdict = ("### ✅ No problems found by the automatic checks\n"
                   "A pharmacist should still confirm before giving it to a patient.")
    return chosen["translation"], chosen["back"], verdict + "\n\n" + "\n\n".join(lines)


demo_examples = sample_df[sample_df.risky].instruction.head(6).tolist()
with gr.Blocks(title="Dose-Safe Translator") as demo:
    gr.Markdown("# 💊 Dose-Safe Translator\nTranslates over-the-counter medicine directions and flags "
                "translations a pharmacist should double-check.\n\n*Prototype – not medical advice.*")
    with gr.Row():
        with gr.Column():
            english_box = gr.Textbox(label="Medicine instruction (English)", lines=3,
                                     value="do not take more than 6 caplets in 24 hours")
            language_box = gr.Dropdown(list(LANGUAGES), value=TEAM_LANGUAGE, label="Translate to")
            safer_box = gr.Checkbox(value=True, label=f"Safer mode: try {N_CANDIDATES} translations "
                                                      "and keep the one that passes the most checks")
            run_button = gr.Button("Translate and check", variant="primary")
        with gr.Column():
            translation_box = gr.Textbox(label="Translation", lines=3)
            back_box = gr.Textbox(label="Translated back into English (for checking)", lines=3)
            report_box = gr.Markdown()
    gr.Examples(examples=[[e] for e in demo_examples], inputs=[english_box])
    run_button.click(check_one, inputs=[english_box, language_box, safer_box],
                     outputs=[translation_box, back_box, report_box])

if not QUICK:
    demo.launch(share=False, debug=False)